# 06 - Robustness Analysis

This notebook reports the full set of predefined variations for the shrinkage GMV allocation.

## 1. Load data and settings

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
current_directory = Path.cwd()
repository_root = current_directory.parent if current_directory.name == "notebooks" else current_directory

if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

from src import active_metrics, load_config, load_monthly_returns, performance_metrics, walk_forward_backtest

In [ ]:
config = load_config(repository_root / "config" / "config.yaml")
monthly_returns = load_monthly_returns(repository_root / config["data_file"])
robustness_config = config["robustness"]

tables_directory = repository_root / "results" / "tables"
figures_directory = repository_root / "results" / "figures"
tables_directory.mkdir(parents=True, exist_ok=True)
figures_directory.mkdir(parents=True, exist_ok=True)

In [ ]:
common_evaluation_start = monthly_returns.index[max(robustness_config["lookback_months"])]

def summarize_backtest(result, start_date=common_evaluation_start):
    result_returns = result["returns"].loc[start_date:]
    portfolio_returns = result_returns["net_return"]
    benchmark_returns = result_returns["benchmark_return"]
    summary = performance_metrics(portfolio_returns)
    summary = pd.concat([summary, active_metrics(portfolio_returns, benchmark_returns)])
    summary["annualized_turnover"] = result_returns["turnover"].mean() * 12
    return summary

## 2. Window, frequency and constraint grid

In [ ]:
robustness_rows = []

for lookback in robustness_config["lookback_months"]:
    for rebalance_months in robustness_config["rebalance_frequency_months"]:
        for max_weight in robustness_config["maximum_weight"]:
            result = walk_forward_backtest(
                monthly_returns,
                method="shrinkage_gmv",
                lookback=lookback,
                rebalance_months=rebalance_months,
                lag=1,
                max_weight=max_weight,
                transaction_cost_bps=config["transaction_cost_bps"],
                factor_columns=config["factor_columns"],
                market_column=config["benchmark"]
            )
            row = summarize_backtest(result)
            row["lookback_months"] = lookback
            row["rebalance_frequency_months"] = rebalance_months
            row["maximum_weight"] = max_weight
            robustness_rows.append(row)

robustness_results = pd.DataFrame(robustness_rows)
robustness_results[["lookback_months", "rebalance_frequency_months"]] = robustness_results[["lookback_months", "rebalance_frequency_months"]].astype(int)
robustness_results

## 3. Cost sensitivity

In [ ]:
cost_rows = []

for transaction_cost_bps in robustness_config["transaction_cost_bps"]:
    result = walk_forward_backtest(
        monthly_returns,
        method="shrinkage_gmv",
        lookback=config["lookback_months"],
        rebalance_months=config["rebalance_frequency_months"],
        lag=1,
        max_weight=config["maximum_weight"],
        transaction_cost_bps=transaction_cost_bps,
        factor_columns=config["factor_columns"],
        market_column=config["benchmark"]
    )
    row = summarize_backtest(result)
    row["transaction_cost_bps"] = transaction_cost_bps
    cost_rows.append(row)

cost_sensitivity = pd.DataFrame(cost_rows)
cost_sensitivity["transaction_cost_bps"] = cost_sensitivity["transaction_cost_bps"].astype(int)
cost_sensitivity

## 4. Equal Weight proxy sensitivity

In [ ]:
factor_sets = {
    "all_factors": config["factor_columns"],
    "without_size_proxy": [
        column for column in config["factor_columns"] if column != "size_proxy"
    ]
}
factor_set_rows = []

for factor_set, columns in factor_sets.items():
    result = walk_forward_backtest(
        monthly_returns,
        method="shrinkage_gmv",
        lookback=config["lookback_months"],
        rebalance_months=config["rebalance_frequency_months"],
        lag=1,
        max_weight=config["maximum_weight"],
        transaction_cost_bps=config["transaction_cost_bps"],
        factor_columns=columns,
        market_column=config["benchmark"]
    )
    row = summarize_backtest(result)
    row["factor_set"] = factor_set
    factor_set_rows.append(row)

factor_set_sensitivity = pd.DataFrame(factor_set_rows)
factor_set_sensitivity

In [ ]:
robustness_results.to_csv(tables_directory / "robustness_grid.csv", index=False)
cost_sensitivity.to_csv(tables_directory / "cost_sensitivity.csv", index=False)
factor_set_sensitivity.to_csv(tables_directory / "factor_set_sensitivity.csv", index=False)

In [ ]:
main_cap_results = robustness_results[
    np.isclose(robustness_results["maximum_weight"], config["maximum_weight"])
]
information_ratio_matrix = main_cap_results.pivot(
    index="lookback_months",
    columns="rebalance_frequency_months",
    values="information_ratio"
)

fig, ax = plt.subplots(figsize=(8, 5))
image = ax.imshow(information_ratio_matrix, aspect="auto", cmap="RdYlGn")
ax.set_xticks(range(len(information_ratio_matrix.columns)))
ax.set_xticklabels([int(value) for value in information_ratio_matrix.columns])
ax.set_yticks(range(len(information_ratio_matrix.index)))
ax.set_yticklabels([int(value) for value in information_ratio_matrix.index])
ax.set_xlabel("Rebalancing frequency in months")
ax.set_ylabel("Lookback in months")
ax.set_title("Shrinkage GMV information ratio")

for row in range(len(information_ratio_matrix.index)):
    for column in range(len(information_ratio_matrix.columns)):
        value = information_ratio_matrix.iloc[row, column]
        ax.text(column, row, f"{value:.2f}", ha="center", va="center")

fig.colorbar(image, ax=ax, label="Information Ratio")
plt.tight_layout()
fig.savefig(figures_directory / "robustness_information_ratio.png", dpi=150)
plt.show()

## Conclusion

The robustness tables report every predefined specification. They are used to assess stability and not to select the historically best combination.